# PAEwall — Dual-Encoder Training

Run via **VSCode Remote SSH + Jupyter** on the GPU server.

**Architecture:**
- Patent tower: `AI-Growth-Lab/PatentSBERTa` (domain-tuned on USPTO)
- Product tower: `sentence-transformers/all-mpnet-base-v2`
- Shared 256-dim projection space
- Loss: InfoNCE with BM25-mined hard negatives

**Workflow:**
1. Run cell 0 — use the upload button to send `pae_bench.parquet` from your local machine to the server
2. Run all remaining cells top to bottom
3. Model artifacts are saved to `models/dual_encoder/` in the repo

## 0. Setup

In [ ]:
# ── Upload pae_bench.parquet from your local machine ─────────────────────────
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

DEST = (Path('..') / 'data' / 'processed' / 'pae_bench.parquet').resolve()
DEST.parent.mkdir(parents=True, exist_ok=True)

upload = widgets.FileUpload(
    accept='.parquet',
    multiple=False,
    description='Upload parquet',
    layout=widgets.Layout(width='250px'),
)
status = widgets.Label(value='⬆ Select pae_bench.parquet to upload it to the server.')

def on_upload(change):
    if not upload.value:
        return
    content = next(iter(upload.value.values()))['content']
    DEST.write_bytes(content)
    status.value = f'✅  Saved to {DEST}  ({len(content)/1e6:.1f} MB)'

upload.observe(on_upload, names='value')
display(widgets.VBox([upload, status]))
# ─────────────────────────────────────────────────────────────────────────────

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import subprocess, sys

def pip(*args):
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f'WARN: pip install {" ".join(args)} failed:\n{result.stderr[-300:]}')
    else:
        print(f'OK: {" ".join(args)}')

pip('sentence-transformers>=2.2.0')
pip('rank-bm25>=0.2.2')
pip('loguru')

# Try GPU faiss first, fall back to CPU
try:
    pip('faiss-gpu')
    import faiss
    print(f'faiss-gpu loaded, GPU count: {faiss.get_num_gpus()}')
except Exception:
    pip('faiss-cpu')
    import faiss
    print('faiss-cpu loaded')

print('All deps ready.')


In [ ]:
from pathlib import Path

# Repo root is one level up from notebooks/
REPO_ROOT  = Path('..').resolve()
DATA_DIR   = REPO_ROOT / 'data' / 'processed'
MODEL_DIR  = REPO_ROOT / 'models' / 'dual_encoder'
OUTPUT_DIR = REPO_ROOT / 'data' / 'outputs'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT :', REPO_ROOT)
print('DATA_DIR  :', DATA_DIR)
print('MODEL_DIR :', MODEL_DIR)
assert (DATA_DIR / 'pae_bench.parquet').exists(), \
    f"pae_bench.parquet not found at {DATA_DIR}. Sync it first (see cell above)."

## 1. Load PAE-Bench

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

bench = pd.read_parquet(DATA_DIR / 'pae_bench.parquet')
print(f'PAE-Bench: {len(bench):,} rows | {bench["patent_id"].nunique():,} patents | {bench["company_name"].nunique():,} companies')
print(f'Verticals:\n{bench["vertical"].value_counts().to_string()}')

patent_ids = bench['patent_id'].unique()
train_ids, test_ids = train_test_split(patent_ids, test_size=0.2, random_state=42)
val_ids,   test_ids = train_test_split(test_ids,   test_size=0.5, random_state=42)

train_df = bench[bench['patent_id'].isin(train_ids)].reset_index(drop=True)
val_df   = bench[bench['patent_id'].isin(val_ids)].reset_index(drop=True)
test_df  = bench[bench['patent_id'].isin(test_ids)].reset_index(drop=True)

product_corpus = (
    bench[['company_name', 'product_description']]
    .drop_duplicates('company_name')
    .reset_index(drop=True)
)
print(f'\nSplit — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'Product corpus: {len(product_corpus):,} unique companies')

## 2. Hard Negative Mining (BM25)

In [ ]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    return re.sub(r'[^\w\s]', ' ', str(text).lower()).split()

print(f'Building BM25 index over {len(product_corpus):,} products...')
bm25 = BM25Okapi([tokenize(t) for t in product_corpus['product_description'].fillna('')])
print('Done.')

def mine_hard_negatives(patent_claims: str, positive_company: str, k: int = 3) -> list[str]:
    scores  = bm25.get_scores(tokenize(patent_claims))
    top_idx = np.argsort(scores)[::-1][:k + 20]
    results = []
    for i in top_idx:
        name = product_corpus['company_name'].iloc[i]
        if name.lower() != positive_company.lower():
            results.append(product_corpus['product_description'].iloc[i])
        if len(results) >= k:
            break
    return results

## 3. Dataset

In [ ]:
from torch.utils.data import Dataset

class PatentProductDataset(Dataset):
    def __init__(self, df: pd.DataFrame, hard_neg_k: int = 3, max_claim_len: int = 2048):
        self.records = []
        for patent_id, group in df.groupby('patent_id'):
            claims      = str(group['patent_claims'].iloc[0] or '')[:max_claim_len]
            pos_company = group['company_name'].iloc[0]
            pos_desc    = str(group['product_description'].iloc[0] or '')[:512]
            hard_negs   = mine_hard_negatives(claims, pos_company, k=hard_neg_k)
            self.records.append({
                'patent_claims': claims,
                'positive':      pos_desc,
                'hard_negatives': [h[:512] for h in hard_negs],
            })

    def __len__(self):        return len(self.records)
    def __getitem__(self, i): return self.records[i]


print('Building datasets (mining hard negatives)...')
train_dataset = PatentProductDataset(train_df)
val_dataset   = PatentProductDataset(val_df)
print(f'Train: {len(train_dataset):,}  Val: {len(val_dataset):,}')

## 4. Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
PATENT_MODEL  = 'AI-Growth-Lab/PatentSBERTa'
PRODUCT_MODEL = 'sentence-transformers/all-mpnet-base-v2'
EMBED_DIM     = 768   # hidden size for both backbones
PROJ_DIM      = 256   # shared projection space
INIT_TEMP     = 0.07  # InfoNCE temperature

def mean_pool(model_output, attention_mask):
    token_emb = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_emb.size()).float()
    return (token_emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


class DualEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.patent_encoder  = AutoModel.from_pretrained(PATENT_MODEL)
        self.product_encoder = AutoModel.from_pretrained(PRODUCT_MODEL)
        self.patent_proj     = nn.Linear(EMBED_DIM, PROJ_DIM)
        self.product_proj    = nn.Linear(EMBED_DIM, PROJ_DIM)
        # Store log(temp) so exp() gives the true temperature and it stays positive
        self.log_temperature = nn.Parameter(torch.tensor(INIT_TEMP).log())

    def encode_patents(self, input_ids, attention_mask):
        out = self.patent_encoder(input_ids=input_ids, attention_mask=attention_mask)
        return F.normalize(self.patent_proj(mean_pool(out, attention_mask)), dim=-1)

    def encode_products(self, input_ids, attention_mask):
        out = self.product_encoder(input_ids=input_ids, attention_mask=attention_mask)
        return F.normalize(self.product_proj(mean_pool(out, attention_mask)), dim=-1)

    def forward(self, patent_inputs, positive_inputs, negative_inputs_list=None):
        p_emb   = self.encode_patents(**patent_inputs)     # (B, D)
        pos_emb = self.encode_products(**positive_inputs)  # (B, D)

        keys = [pos_emb]
        if negative_inputs_list:
            for neg_inp in negative_inputs_list:
                keys.append(self.encode_products(**neg_inp))

        keys     = torch.stack(keys, dim=1)      # (B, 1+K, D)
        B, NK, D = keys.shape
        temp     = self.log_temperature.exp().clamp(min=0.01, max=1.0)
        logits   = (p_emb @ keys.view(B * NK, D).T) / temp  # (B, B*(1+K))
        labels   = torch.arange(B, device=logits.device) * NK
        return F.cross_entropy(logits, labels)


print(f'Loading encoders on {DEVICE}...')
model           = DualEncoder().to(DEVICE)
patent_tok      = AutoTokenizer.from_pretrained(PATENT_MODEL)
product_tok     = AutoTokenizer.from_pretrained(PRODUCT_MODEL)
print(f'Ready. Initial temperature = {INIT_TEMP}')


## 5. Training

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

BATCH_SIZE   = 16
EPOCHS       = 5
LR           = 2e-5
MAX_PAT_LEN  = 512
MAX_PROD_LEN = 256
HARD_NEG_K   = 3

def tok_batch(texts, tokenizer, max_len):
    return tokenizer(
        texts, padding=True, truncation=True,
        max_length=max_len, return_tensors='pt'
    ).to(DEVICE)

def collate(batch):
    return {
        'patent_inputs':   tok_batch([b['patent_claims'] for b in batch], patent_tok,  MAX_PAT_LEN),
        'positive_inputs': tok_batch([b['positive']      for b in batch], product_tok, MAX_PROD_LEN),
        'negative_inputs_list': [
            tok_batch(
                [b['hard_negatives'][k] if k < len(b['hard_negatives']) else '' for b in batch],
                product_tok, MAX_PROD_LEN
            )
            for k in range(HARD_NEG_K)
        ],
    }

# num_workers=0 required in Jupyter+CUDA — forked workers can't re-init CUDA
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate, num_workers=0)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))
print(f'{len(train_loader)} batches/epoch × {EPOCHS} epochs')


In [ ]:
best_val_loss = float('inf')
history = []

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_losses = []
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} train'):
        optimizer.zero_grad()
        loss = model(**batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_losses.append(loss.item())

    # Validate
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS} val  '):
            val_losses.append(model(**batch).item())

    tl, vl = np.mean(train_losses), np.mean(val_losses)
    history.append({'epoch': epoch, 'train_loss': float(tl), 'val_loss': float(vl)})
    print(f'Epoch {epoch}: train={tl:.4f}  val={vl:.4f}')

    if vl < best_val_loss:
        best_val_loss = vl
        # Save only tensors + plain-Python config so weights_only=True works on load
        torch.save({
            'epoch': int(epoch),
            'val_loss': float(vl),
            'model_state': model.state_dict(),
            'config': dict(patent_model=PATENT_MODEL, product_model=PRODUCT_MODEL,
                           batch_size=int(BATCH_SIZE), lr=float(LR),
                           hard_neg_k=int(HARD_NEG_K), proj_dim=int(PROJ_DIM)),
        }, MODEL_DIR / 'best_model.pt')
        print(f'  -> Saved best checkpoint (val={vl:.4f})')


## 6. Evaluation — Recall@K, MRR, nDCG@10

In [ ]:
try:
    import faiss
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'])
    import faiss

# weights_only=False: optimizer_state contains numpy scalars (AdamW step counts)
# safe here because we wrote this checkpoint ourselves
ckpt = torch.load(MODEL_DIR / 'best_model.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded epoch {ckpt['epoch']} (val={ckpt['val_loss']:.4f})")


@torch.no_grad()
def encode_products(corpus: pd.DataFrame, batch_size: int = 128) -> np.ndarray:
    embs = []
    texts = corpus['product_description'].fillna('').tolist()
    for i in range(0, len(texts), batch_size):
        inp = tok_batch(texts[i:i+batch_size], product_tok, MAX_PROD_LEN)
        embs.append(model.encode_products(**inp).cpu().float().numpy())
    return np.vstack(embs)


@torch.no_grad()
def encode_patent(claims: str) -> np.ndarray:
    inp = tok_batch([claims], patent_tok, MAX_PAT_LEN)
    return model.encode_patents(**inp).cpu().float().numpy()


def build_index(embs: np.ndarray) -> faiss.Index:
    embs = embs.copy()
    faiss.normalize_L2(embs)
    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs)
    return idx


def eval_retrieval(df: pd.DataFrame, corpus: pd.DataFrame, index: faiss.Index, K: int = 50) -> dict:
    r10, r50, mrr, ndcg = [], [], [], []
    for pid, grp in df.groupby('patent_id'):
        claims   = str(grp['patent_claims'].iloc[0] or '')
        relevant = set(grp['company_name'].str.lower())
        q        = encode_patent(claims)
        faiss.normalize_L2(q)
        _, idxs  = index.search(q, K)
        retrieved = [corpus['company_name'].iloc[i].lower() for i in idxs[0]]

        r10.append(len(set(retrieved[:10]) & relevant) / max(len(relevant), 1))
        r50.append(len(set(retrieved[:50]) & relevant) / max(len(relevant), 1))
        mrr.append(next((1/(r+1) for r, c in enumerate(retrieved) if c in relevant), 0.0))
        dcg  = sum(1/np.log2(r+2) for r,c in enumerate(retrieved[:10]) if c in relevant)
        idcg = sum(1/np.log2(i+2) for i in range(min(len(relevant), 10)))
        ndcg.append(dcg / max(idcg, 1e-10))

    return {'Recall@10': np.mean(r10), 'Recall@50': np.mean(r50),
            'MRR': np.mean(mrr), 'nDCG@10': np.mean(ndcg), 'n': len(r10)}


print('Encoding product corpus...')
prod_embs = encode_products(product_corpus)
index     = build_index(prod_embs.copy())

print('Evaluating on test set...')
test_metrics = eval_retrieval(test_df, product_corpus, index)

print('\n=== Dual-Encoder (text-only) — Test Results ===')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')


## 7. Per-Vertical Breakdown + Ablation Record

In [ ]:
import json

per_vertical = {}
for vertical in test_df['vertical'].unique():
    subset = test_df[test_df['vertical'] == vertical]
    if len(subset) < 5:
        continue
    vm = eval_retrieval(subset, product_corpus, index)
    per_vertical[vertical] = {k: float(v) for k, v in vm.items()}
    print(f'  [{vertical:20s}] R@10={vm["Recall@10"]:.4f}  MRR={vm["MRR"]:.4f}  n={vm["n"]}')

results = {
    'model': 'dual_encoder_text_only',
    'config': ckpt['config'],
    'overall': {k: float(v) for k, v in test_metrics.items()},
    'per_vertical': per_vertical,
    'training_history': history,
}

out_path = OUTPUT_DIR / 'dual_encoder_text_only_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved to {out_path}')

## 8. Save FAISS Index + Corpus for Inference App

In [ ]:
import pickle

faiss.write_index(index, str(MODEL_DIR / 'product_index.faiss'))
product_corpus.to_parquet(MODEL_DIR / 'product_corpus.parquet', index=False)
np.save(MODEL_DIR / 'product_embeddings.npy', prod_embs)

# product_meta.pkl — required by RetrievalEngine in app.py
meta = {
    'ids':          [str(i) for i in range(len(product_corpus))],
    'names':        product_corpus['company_name'].tolist(),
    'descriptions': product_corpus['product_description'].fillna('').tolist(),
}
with open(MODEL_DIR / 'product_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print('Saved inference artifacts:')
for f in sorted(MODEL_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')


## 9. Download Artifacts to Local Machine

In [ ]:
# ── Package inference artifacts and download ─────────────────────────────
import tarfile, io, os
from pathlib import Path
from IPython.display import display, FileLink, HTML

MODEL_DIR = Path('..').resolve() / 'models' / 'dual_encoder'

# 1. Save a projection-only checkpoint (~1.5 MB) for CPU inference
#    The full best_model.pt is 877 MB (both backbones). The lite version
#    stores only the projection heads + temperature — enough if the app
#    re-loads the backbone from HuggingFace at startup.
lite_ckpt = {
    'patent_proj':  model.patent_proj.state_dict(),
    'product_proj': model.product_proj.state_dict(),
    'log_temperature': model.log_temperature.detach().cpu(),
    'config': ckpt['config'],
    'epoch': ckpt['epoch'],
    'val_loss': ckpt['val_loss'],
}
import torch
torch.save(lite_ckpt, MODEL_DIR / 'proj_only.pt')
print(f"proj_only.pt saved ({(MODEL_DIR/'proj_only.pt').stat().st_size/1e6:.1f} MB)")

# 2. Bundle the small inference files into a single tar.gz
BUNDLE = MODEL_DIR / 'dual_encoder_inference.tar.gz'
small_files = ['product_index.faiss', 'product_meta.pkl',
               'product_corpus.parquet', 'proj_only.pt']
with tarfile.open(BUNDLE, 'w:gz') as tar:
    for fname in small_files:
        p = MODEL_DIR / fname
        if p.exists():
            tar.add(p, arcname=fname)
            print(f"  + {fname}  ({p.stat().st_size/1e6:.2f} MB)")
print(f"\nBundle: {BUNDLE}  ({BUNDLE.stat().st_size/1e6:.1f} MB)")

# 3. Clickable download link (works in Jupyter / VSCode notebooks)
rel = os.path.relpath(BUNDLE)
display(FileLink(rel, result_html_prefix='⬇  Download bundle: '))

# 4. SCP commands for best_model.pt (877 MB — download separately)
import socket
hostname = socket.gethostname()
remote_model = MODEL_DIR / 'best_model.pt'
local_dest = 'models/dual_encoder/'
print('\n── Manual download commands (run on YOUR local machine) ──')
print(f'mkdir -p {local_dest}')
print(f'scp {hostname}:{BUNDLE} {local_dest}')
print(f'scp {hostname}:{remote_model} {local_dest}')
print(f'# Then extract: cd models/dual_encoder && tar xzf dual_encoder_inference.tar.gz')


In [ ]:
# ── Upload model files to Google Drive ───────────────────────────────────
# STEP 1 (in your LOCAL browser):
#   Go to https://developers.google.com/oauthplayground/
#   Top-right gear icon → check 'Use your own OAuth credentials' (leave defaults)
#   In left panel: scroll to 'Drive API v3' → select '.../auth/drive.file' → Authorize APIs
#   Sign in with your Google account → Step 2: click 'Exchange authorization code for tokens'
#   Copy the 'Access token' (starts with REDACTED_TOKEN)
#
# STEP 2: paste it below and run this cell

ACCESS_TOKEN = "REDACTED_TOKEN"  # <-- replace this

# ─────────────────────────────────────────────────────────────────────────
import requests
from pathlib import Path

MODEL_DIR = Path('..').resolve() / 'models' / 'dual_encoder'

FILES_TO_UPLOAD = [
    MODEL_DIR / 'best_model.pt',
    MODEL_DIR / 'dual_encoder_inference.tar.gz',
]

HEADERS = {'Authorization': f'Bearer {ACCESS_TOKEN}'}

# Create a folder in Google Drive
folder_meta = {'name': 'PAEwall_dual_encoder', 'mimeType': 'application/vnd.google-apps.folder'}
r = requests.post('https://www.googleapis.com/drive/v3/files',
                  headers=HEADERS, json=folder_meta)
folder_id = r.json().get('id')
print(f'Drive folder id: {folder_id}')

for fpath in FILES_TO_UPLOAD:
    if not fpath.exists():
        print(f'SKIP (not found): {fpath.name}')
        continue
    size_mb = fpath.stat().st_size / 1e6
    print(f'Uploading {fpath.name}  ({size_mb:.0f} MB) ...')

    # Use resumable upload for large files
    init_r = requests.post(
        'https://www.googleapis.com/upload/drive/v3/files?uploadType=resumable',
        headers={**HEADERS, 'Content-Type': 'application/json',
                 'X-Upload-Content-Type': 'application/octet-stream',
                 'X-Upload-Content-Length': str(fpath.stat().st_size)},
        json={'name': fpath.name, 'parents': [folder_id]},
    )
    upload_url = init_r.headers.get('Location')

    with open(fpath, 'rb') as f:
        up_r = requests.put(upload_url, data=f,
                            headers={'Content-Type': 'application/octet-stream'})
    if up_r.status_code in (200, 201):
        fid = up_r.json().get('id')
        print(f'  ✅  {fpath.name} → https://drive.google.com/file/d/{fid}/view')
    else:
        print(f'  ❌  {fpath.name} failed: {up_r.status_code} {up_r.text[:200]}')

print('\nDone. Open Google Drive → PAEwall_dual_encoder folder to download.')


In [ ]:
# ── Option A: base64-encode small inference files (no storage needed) ────
# This encodes the tar.gz (~3 MB) as text you can copy-paste to your local.
import base64, sys
from pathlib import Path

MODEL_DIR = Path('..').resolve() / 'models' / 'dual_encoder'
BUNDLE    = MODEL_DIR / 'dual_encoder_inference.tar.gz'

if not BUNDLE.exists():
    print('Run cell 8 (Save FAISS Index) and cell 9 (Download Artifacts) first.')
else:
    data    = BUNDLE.read_bytes()
    encoded = base64.b64encode(data).decode()
    print(f'File: {BUNDLE.name}  ({len(data)/1e6:.2f} MB  →  {len(encoded)} base64 chars)')
    print('--- BEGIN BASE64 ---')
    # Print in 76-char lines (standard PEM style)
    for i in range(0, len(encoded), 76):
        print(encoded[i:i+76])
    print('--- END BASE64 ---')
    print('\nCopy everything between BEGIN/END and run the decode script locally.')


### Decode on your LOCAL machine

1. Copy everything between `--- BEGIN BASE64 ---` and `--- END BASE64 ---` from the output above.
2. Save it to a file called `bundle.b64` on your local machine.
3. Run:
```bash
python3 -c "
import base64, pathlib
pathlib.Path('models/dual_encoder').mkdir(parents=True, exist_ok=True)
data = base64.b64decode(open('bundle.b64').read().replace('\\n',''))
open('models/dual_encoder/dual_encoder_inference.tar.gz','wb').write(data)
print('Saved', len(data), 'bytes')
"
cd models/dual_encoder && tar xzf dual_encoder_inference.tar.gz
```

**For `best_model.pt` (877 MB):**  
In VSCode (Remote SSH window) → Explorer sidebar → navigate to `/models/dual_encoder/best_model.pt` → **right-click → Download...** — this uses the SSH tunnel directly, no browser needed.
